[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Tuning


## What you will be able to do

Choose a boosted model's settings by cross-validation that keeps each station's days together, with
scikit-learn's searches and with each library's own `cv`, and measure the model you chose once on the
test stations.

## The idea

### The problem

A boosted model has settings that are fixed before it trains: a learning rate, a number of leaves, a depth. Each
library has dozens of settings, and the values that suit these stations may not suit another
table. Finding them means training many candidates and keeping the one that scores best,
which raises the question of what to score them on.

The test stations cannot be the judge. A model chosen because it scored best on the test stations
has been fitted to them through that choice, and its test score stops being an honest estimate of
how it will do on stations nobody has seen. The 50 validation stations are better, but a single set
of 50 stations is a noisy judge, and a difference between two candidates there can come from which
stations happened to be in it.

### A formal definition

> **Hyperparameter tuning** is choosing the settings of a model that are fixed before it trains, such
> as its number of leaves or its learning rate, by comparing the scores each candidate earns on data
> it was not trained on. **Grouped cross-validation** splits the rows into folds by a group, here the
> station, so that every row of a group falls in the same fold, trains on all folds but one, scores on
> the one left out, and repeats until every fold has been left out once.

### Why it works that way

Five folds of the 300 training stations give five scores for each candidate, each from 60 stations
the candidate never trained on. Their mean is steadier than one validation score, and because every
candidate is scored on the same folds, two candidates can be compared fold by fold to see whether one
does better on every set of stations or only on some.

The grouping matters because every station has a warmth or coldness of its own. If the folds split
rows instead of stations, a fold would score days of stations whose other days were in training, and
the model would already know those stations. The score would describe stations it has seen, which is
not the question.

A grid search tries every combination of a few values for each setting. A random search draws a fixed number of combinations from ranges you give, which covers more
values of each setting for the same number of fits. Either way, the best of many scores is a little
lucky. The chosen model is refitted on all the training stations so that it learns from every one of
them, and is then measured once on the test stations, which took no part in choosing it and so give an
estimate free of that luck.

### Where this shows up

The **scikit-learn, Deep Dive** guide searched settings the same way, with scikit-learn's own models.
The **Early Stopping** notebook let the validation stations choose the number of trees, and the
libraries' `lgb.cv` and `xgb.cv` let folds choose it instead. The **A Frost Forecast** notebook
chooses its settings on the validation stations.

### What this notebook covers

- grouped folds of the training stations, and one model cross-validated on them
- a grid search for LightGBM and a random search for XGBoost
- the number of trees chosen by each library's own cross-validation
- the chosen models, refitted and measured once on the test stations
- four mistakes, among them folds that split a station's days

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
from sklearn.model_selection import GroupKFold

station = ["A", "A", "B", "B", "B", "C", "C"]
for train_rows, test_rows in GroupKFold(n_splits=3, shuffle=True, random_state=0).split(station, groups=station):
    print("train", [station[i] for i in train_rows], "| test", [station[i] for i in test_rows])
```

```
train ['A', 'A', 'B', 'B', 'B'] | test ['C', 'C']
train ['A', 'A', 'C', 'C'] | test ['B', 'B', 'B']
train ['B', 'B', 'B', 'C', 'C'] | test ['A', 'A']
```

Each fold leaves out one station whole, so no row of a left-out station is ever in training at the
same time.

## Setup

Two libraries for the table, the two boosting libraries, three ways to draw random settings, and the
parts of scikit-learn that search and score.

- `numpy` and `pandas` build the table and the tables of results
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `loguniform`, `randint` and `uniform` from `scipy.stats` describe the ranges a random search draws
  from
- `GroupKFold` makes folds by station, and `KFold`, which splits rows, is used once in the Common
  errors
- `cross_val_score`, `GridSearchCV` and `RandomizedSearchCV` cross-validate one model and search many
- `mean_squared_error` scores a model on the test stations
- `warnings` catches the warnings XGBoost gives in the Common errors

`station_days` makes the same 20,000 made-up station-days as the rest of the guide, from a fixed
seed, so the table is the same on every run, and the cell splits the stations into 300 for training,
50 for validation and 50 for testing. This notebook tunes on the training stations alone, measures
the chosen models on the test stations, and leaves the validation stations to one of the Common
errors and a task in Your turn.

In [1]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, RandomizedSearchCV, cross_val_score

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", train["station"].nunique(), "training stations,",
      len(train), "rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 300 training stations, 15000 rows


## Worked examples

### Folds made of whole stations

`groups` gives each row its station, and `GroupKFold` keeps a station's rows in one fold. Without
`shuffle`, `GroupKFold` gives the stations to the folds largest first. Every training station has 50
days, so the order they go in comes from how NumPy's `argsort` orders equal values, which NumPy leaves
unspecified, and NumPy 2.1.3 and 2.4.6 made different folds from it. With `shuffle=True` and a `random_state`, it splits a
seeded permutation of the stations into five instead, and no sort of equal values decides the folds.

In [2]:
X, y, stations = train[FEATURES], train["tmin"], train["station"]
folds = GroupKFold(n_splits=5, shuffle=True, random_state=0)

for number, (fit_rows, score_rows) in enumerate(folds.split(X, y, groups=stations), 1):
    shared = set(stations.iloc[fit_rows]) & set(stations.iloc[score_rows])
    print("fold", number, "|", stations.iloc[score_rows].nunique(), "stations scored,",
          len(score_rows), "rows | stations in both halves:", len(shared))

fold 1 | 60 stations scored, 3000 rows | stations in both halves: 0
fold 2 | 60 stations scored, 3000 rows | stations in both halves: 0
fold 3 | 60 stations scored, 3000 rows | stations in both halves: 0
fold 4 | 60 stations scored, 3000 rows | stations in both halves: 0
fold 5 | 60 stations scored, 3000 rows | stations in both halves: 0


Every fold scores 60 stations, 3,000 rows, and no station has rows on both sides of any fold.

### Cross-validating a single model

`cross_val_score` trains a copy of the model on four folds and scores it on the fifth, five times.
scikit-learn maximizes its scores, so a mean squared error comes back negative, as
`"neg_mean_squared_error"`, and a minus sign makes it positive.

In [3]:
model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
scores = -cross_val_score(model, X, y, groups=stations, cv=folds, scoring="neg_mean_squared_error")
print("each fold:", scores.round(3))
print("mean", round(scores.mean(), 3), "| spread", round(scores.std(), 3))

each fold: [2.731 2.488 2.749 2.632 3.363]
mean 2.792 | spread 0.3


The five folds score from 2.488 to 3.363 around a mean of 2.792, with a standard deviation, the
`spread`, of 0.3. Every fold's model has the same settings, so the spread comes from which 60
stations each fold holds, and those stations are the same for every candidate below.

### A grid search for LightGBM

`GridSearchCV` cross-validates every combination of the values it is given, here three numbers of
leaves and three smallest leaf sizes, nine candidates and 45 fits. `cv_results_` holds every
candidate's mean and spread.

In [4]:
grid = GridSearchCV(
    lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1),
    {"num_leaves": [7, 31, 127], "min_child_samples": [5, 20, 80]},
    cv=folds, scoring="neg_mean_squared_error",
)
grid.fit(X, y, groups=stations)

results = pd.DataFrame(grid.cv_results_)
table = pd.DataFrame({
    "num_leaves": results["param_num_leaves"], "min_child_samples": results["param_min_child_samples"],
    "mean": -results["mean_test_score"], "spread": results["std_test_score"],
})
print(table.sort_values(["mean", "num_leaves"], kind="stable").round(3).to_string(index=False))
print("chosen:", grid.best_params_)

 num_leaves  min_child_samples  mean  spread
          7                 80 2.723   0.228
         31                 80 2.738   0.268
          7                 20 2.749   0.280
          7                  5 2.773   0.266
         31                 20 2.792   0.300
         31                  5 2.806   0.291
        127                 80 2.841   0.276
        127                 20 2.934   0.306
        127                  5 2.970   0.383
chosen: {'min_child_samples': 80, 'num_leaves': 7}


The grid chooses 7 leaves and a smallest leaf of 80 rows, at 2.723, and the next five candidates are
within 0.09 of it. Every candidate is scored on the same five folds, and a fold of hard stations raises
every candidate's loss together, so a difference between two candidates is read fold by fold. The cell
below finds each candidate's hardest and easiest fold, and prints the losses of the three candidates
with a smallest leaf of 80.

In [5]:
per_fold = pd.DataFrame({f"fold {k}": -results[f"split{k - 1}_test_score"] for k in range(1, 6)})
per_fold.index = [f"{leaves} leaves, {size}"
                  for leaves, size in zip(results["param_num_leaves"], results["param_min_child_samples"])]
print("highest loss on:", sorted(set(per_fold.idxmax(axis=1))),
      "| lowest on:", sorted(set(per_fold.idxmin(axis=1))))
print(per_fold.loc[["7 leaves, 80", "31 leaves, 80", "127 leaves, 80"]].round(3).to_string())

highest loss on: ['fold 5'] | lowest on: ['fold 2']
                fold 1  fold 2  fold 3  fold 4  fold 5
7 leaves, 80     2.584   2.488   2.761   2.639   3.145
31 leaves, 80    2.692   2.445   2.736   2.581   3.236
127 leaves, 80   2.704   2.522   2.936   2.717   3.326


All nine candidates have their highest loss on fold 5 and their lowest on fold 2. Against 31 leaves
with the same smallest leaf, the chosen candidate is lower on folds 1 and 5 and higher on folds 2, 3
and 4, so other folds could put 31 leaves first. With 127 leaves the loss is higher than both on all
five folds. The three candidates with 127 leaves are the three worst in the table, from 2.841 to 2.970,
which is the clearest thing the grid shows.

### A random search for XGBoost

`RandomizedSearchCV` draws `n_iter` candidates from the ranges it is given. `randint(2, 9)` draws a
whole number from 2 to 8, `loguniform(0.02, 0.3)` a learning rate spread evenly on a log scale, and
`uniform(0.5, 0.5)` a share from 0.5 to 1.0, for the rows and the columns each tree sees.
`random_state` fixes the draws, and the model's own `random_state` fixes its sampling.

In [6]:
search = RandomizedSearchCV(
    xgb.XGBRegressor(n_estimators=200, random_state=0),
    {"max_depth": randint(2, 9), "learning_rate": loguniform(0.02, 0.3),
     "subsample": uniform(0.5, 0.5), "colsample_bytree": uniform(0.5, 0.5)},
    n_iter=8, cv=folds, scoring="neg_mean_squared_error", random_state=0,
)
search.fit(X, y, groups=stations)

drawn = pd.DataFrame(search.cv_results_)
table = pd.DataFrame({
    "max_depth": drawn["param_max_depth"], "learning_rate": drawn["param_learning_rate"].astype(float),
    "subsample": drawn["param_subsample"].astype(float), "colsample_bytree": drawn["param_colsample_bytree"].astype(float),
    "mean": -drawn["mean_test_score"], "spread": drawn["std_test_score"],
})
print(table.sort_values("mean", kind="stable").round(3).to_string(index=False))
print("chosen: max_depth", search.best_params_["max_depth"], "| learning_rate", round(float(search.best_params_["learning_rate"]), 3))

 max_depth  learning_rate  subsample  colsample_bytree  mean  spread
         5          0.070      0.839             0.900 2.704   0.329
         4          0.108      0.719             0.924 2.727   0.269
         5          0.139      0.929             0.774 2.755   0.290
         3          0.097      0.972             0.860 2.826   0.261
         3          0.073      0.963             0.906 2.889   0.314
         4          0.272      0.739             0.946 2.935   0.278
         5          0.025      0.684             0.536 3.452   0.348
         2          0.029      0.989             0.979 5.285   0.566
chosen: max_depth 5 | learning_rate 0.07


The eight draws score from 2.704 to 5.285, and `best_params_`, printed last, holds the draw with the
lowest mean, depth 5 at a learning rate of 0.070. The second, depth 4 at 0.108, is 0.023 behind it,
and the cell below compares the two fold by fold.

In [7]:
by_fold = pd.DataFrame({f"fold {k}": -drawn[f"split{k - 1}_test_score"] for k in range(1, 6)})
by_fold.index = [f"depth {depth}, rate {rate:.3f}"
                 for depth, rate in zip(table["max_depth"], table["learning_rate"])]
print(by_fold.loc[["depth 5, rate 0.070", "depth 4, rate 0.108"]].round(3).to_string())

                     fold 1  fold 2  fold 3  fold 4  fold 5
depth 5, rate 0.070   2.705   2.371   2.575   2.541   3.326
depth 4, rate 0.108   2.652   2.449   2.680   2.613   3.240


The draw at depth 5 is lower on folds 2, 3 and 4 and higher on folds 1 and 5, so other folds could
order the two differently. The two worst draws both pair a learning rate below 0.03 with 200 trees, a
slow rate of the kind the **Learning Rate and Trees** notebook pairs with more trees.

### Folds that choose the number of trees

Both libraries have a cross-validation of their own, which trains every fold one round at a time
and can stop early when the mean score across the folds stops improving. Given `folds`, the same
station folds as above, each returns the scores round by round, up to the best round.

In [8]:
station_folds = list(folds.split(X, y, groups=stations))

lightgbm_cv = lgb.cv(
    {"objective": "regression", "learning_rate": 0.1, "num_leaves": 31, "seed": 0, "verbose": -1},
    lgb.Dataset(X, y), num_boost_round=1000, folds=station_folds,
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
print("LightGBM:", len(lightgbm_cv["valid l2-mean"]), "rounds | mean", round(lightgbm_cv["valid l2-mean"][-1], 3),
      "| spread", round(lightgbm_cv["valid l2-stdv"][-1], 3))

xgboost_cv = xgb.cv(
    {"objective": "reg:squarederror", "eta": 0.1, "max_depth": 6, "seed": 0},
    xgb.DMatrix(X, y, enable_categorical=True), num_boost_round=1000, folds=station_folds,
    early_stopping_rounds=50, metrics="rmse",
)
print("XGBoost:", len(xgboost_cv), "rounds | mean RMSE", round(xgboost_cv["test-rmse-mean"].iloc[-1], 3),
      "| spread", round(xgboost_cv["test-rmse-std"].iloc[-1], 3))

LightGBM: 150 rounds | mean 2.764 | spread 0.302
XGBoost: 116 rounds | mean RMSE 1.669 | spread 0.086


LightGBM's folds stop improving at 150 rounds, with a mean squared error of 2.764, and XGBoost's at
116 rounds. XGBoost was asked for `rmse`, the square root of each fold's mean squared error. Its mean,
1.669, squared is 2.79, a little below the folds' own mean squared error, since squaring an average of
roots gives less than averaging the squares. Either way it is close to LightGBM's 2.764. Each library
trained on the station folds, so these are numbers of trees chosen without the test stations.

### The chosen models measured once on the test stations

`GridSearchCV` and `RandomizedSearchCV` refit the chosen candidate on all 300 training stations when
they finish, and `predict` uses that refitted model. Each is measured once on the test stations.

In [9]:
print("LightGBM, chosen by the grid:", round(mean_squared_error(test["tmin"], grid.predict(test[FEATURES])), 3))
print("XGBoost, chosen by the random search:", round(mean_squared_error(test["tmin"], search.predict(test[FEATURES])), 3))

LightGBM, chosen by the grid: 2.608
XGBoost, chosen by the random search: 2.552


On the test stations the grid's LightGBM model scores 2.608 and the random search's XGBoost model
2.552, both below their cross-validation scores of 2.723 and 2.704. The refitted models trained on all
300 stations, where a fold's model trained on 240, and these 50 test stations may also be easier than
the folds. These scores cannot separate the two. The two searches tried different settings in different numbers,
and their two scores describe two chosen models, which leaves the libraries unranked.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/12-tuning-solutions.ipynb).

**1.** Cross-validate an `XGBRegressor` with 200 trees and a learning rate of 0.1 on the station folds, and print each fold's mean squared error and their mean.

In [10]:
# your code here

**2.** Run a grid search for LightGBM over `num_leaves` of 15 and 31 and `learning_rate` of 0.05 and 0.1, with 200 trees, and print the chosen settings and their mean score. Name it `rate_grid`, so that `grid` still holds the worked examples' search.

In [11]:
# your code here

**3.** Run a random search of 6 candidates for LightGBM over `num_leaves` from `randint(4, 128)` and `min_child_samples` from `randint(5, 100)`, with `random_state=0`, and print the candidates sorted by their mean score.

In [12]:
# your code here

**4.** Run `lgb.cv` on the station folds with a learning rate of 0.05 in place of 0.1, and print how many rounds it keeps and its mean score.

In [13]:
# your code here

**5.** Score the grid's chosen model, `grid.best_estimator_`, and an `LGBMRegressor` with its default settings on the validation stations, which took no part in the search.

In [14]:
# your code here

## Common errors

### `ValueError: The 'groups' parameter should not be None.`

In [15]:
GridSearchCV(lgb.LGBMRegressor(verbose=-1), {"num_leaves": [7, 31]}, cv=folds).fit(X, y)

ValueError: The 'groups' parameter should not be None.

`GroupKFold` needs every row's group, and a search passes the groups on only when its `fit` is given
`groups=`. Pass `groups=stations` to `fit`, as the worked examples do.

### `ValueError: Supported target types are: ('binary', 'multiclass'). Got 'continuous' instead.`

In [16]:
lgb.cv({"objective": "regression", "verbose": -1}, lgb.Dataset(X, y), num_boost_round=50)

ValueError: Supported target types are: ('binary', 'multiclass'). Got 'continuous' instead.

`lgb.cv` makes five folds of its own when it is given none, and by default it makes them stratified,
with the same share of each class in every fold. A temperature has no classes to share out. Pass
`folds=` with the station folds, as the worked examples do, since `stratified=False` would make folds
of rows, the next mistake.

### No error, and a cross-validation score that is too good

In [17]:
same_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
by_rows = -cross_val_score(same_model, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=0),
                           scoring="neg_mean_squared_error")
by_stations = -cross_val_score(same_model, X, y, groups=stations, cv=folds, scoring="neg_mean_squared_error")
same_model.fit(X, y)
print("folds of rows:", round(by_rows.mean(), 3))
print("folds of stations:", round(by_stations.mean(), 3))
print("validation stations:", round(mean_squared_error(valid["tmin"], same_model.predict(valid[FEATURES])), 3))

folds of rows: 1.794
folds of stations: 2.792
validation stations: 2.581


The same model scores 1.794 in folds of rows, 2.792 in folds of stations, and 2.581 on the validation
stations. With rows shuffled into folds, nearly every scored day belongs to a station whose other days
were in training, so the model has already learned that station's warmth or coldness, and the score
describes stations it has partly seen. Folds of stations score it on stations it has never seen, as
the validation stations do.

### No error, and a misspelled setting that changes nothing

In [18]:
misspelled = GridSearchCV(lgb.LGBMRegressor(n_estimators=100, random_state=0, verbose=-1),
                          {"num_leafs": [7, 31, 127]}, cv=folds, scoring="neg_mean_squared_error")
misspelled.fit(X, y, groups=stations)
print("LightGBM, num_leafs:", (-misspelled.cv_results_["mean_test_score"]).round(3))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    misspelled_xgb = GridSearchCV(xgb.XGBRegressor(n_estimators=100, random_state=0),
                                  {"num_leaves": [7, 31]}, cv=folds, scoring="neg_mean_squared_error")
    misspelled_xgb.fit(X, y, groups=stations)
print("XGBoost, num_leaves:", (-misspelled_xgb.cv_results_["mean_test_score"]).round(3))
messages = {str(w.message).split("\n", 1)[1].strip() for w in caught if "are not used" in str(w.message)}
print(len(caught), "warnings, all saying:", messages)
print("in get_params():", {name: name in lgb.LGBMRegressor().get_params() for name in ("num_leafs", "num_leaves", "max_cat_to_onehot")},
      "| given num_leafs=7:", "num_leafs" in lgb.LGBMRegressor(num_leafs=7).get_params())

LightGBM, num_leafs: [2.788 2.788 2.788]
XGBoost, num_leaves: [3.073 3.073]
11 warnings, all saying: {'Parameters: { "num_leaves" } are not used.'}
in get_params(): {'num_leafs': False, 'num_leaves': True, 'max_cat_to_onehot': False} | given num_leafs=7: True


LightGBM trained with `num_leafs` and ignored it, so the three candidates are one model with one score,
2.788, and with `verbose=-1`, which the search's model was given, it said nothing. XGBoost
trains too, and warns at every fit that `num_leaves`, LightGBM's name for what XGBoost calls
`max_leaves`, was not used, 11 times here, once for each of five
folds of two candidates and once for the refit. Neither stops the search. LightGBM does report the
name, at a verbosity that shows warnings, as the next cell shows with `verbose=0`.

In [19]:
shown = lgb.LGBMRegressor(n_estimators=5, num_leafs=7, verbose=0).fit(X, y)

[LightGBM] [Warning] Unknown parameter: num_leafs
[LightGBM] [Warning] Unknown parameter: num_leafs


LightGBM prints `Unknown parameter: num_leafs` at `verbose=0` and nothing at `verbose=-1`. To check a
grid's names before searching, fit one model with them at `verbose=0`, as the cell above does, and
read its warnings. `get_params()` cannot settle it for LightGBM. A new `LGBMRegressor()` lists only
the class's own arguments, so `max_cat_to_onehot`, which the **Categorical Features** notebook passes to
`LGBMRegressor`, is missing from it. A model given any other keyword lists that keyword back whether
LightGBM knows it or not, as the last line of the misspelled grid's cell shows for `num_leafs=7`.

## Recap

- You can make folds of whole stations with `GroupKFold`, and pass `groups` to every search that uses
  them.
- You can cross-validate one model with `cross_val_score`, and read the spread of its fold scores.
- You can search settings with `GridSearchCV` and `RandomizedSearchCV`, and let `lgb.cv` and `xgb.cv`
  choose the number of trees.
- You can measure the model a search chose once, on test stations that took no part in choosing it.

## What is next

The **Reading What They Learned** notebook asks a trained model which features it used, by its
importances and by the contribution of each feature to each prediction.

---

&#8592; **Previous:** [Imbalanced Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/11-imbalanced-classes.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Reading What They Learned](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/13-reading-what-they-learned.ipynb) &#8594;
